In [1]:
# imports and env
import base64
from dotenv import load_dotenv
from langchain_unstructured.document_loaders import UnstructuredLoader
from langchain_openai import ChatOpenAI
from langchain_experimental.open_clip import OpenCLIPEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import HumanMessage

load_dotenv()

/home/rikesh/Rikesh/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
import open_clip

open_clip.list_pretrained()

[('RN50', 'openai'),
 ('RN50', 'yfcc15m'),
 ('RN50', 'cc12m'),
 ('RN101', 'openai'),
 ('RN101', 'yfcc15m'),
 ('RN50x4', 'openai'),
 ('RN50x16', 'openai'),
 ('RN50x64', 'openai'),
 ('ViT-B-32', 'openai'),
 ('ViT-B-32', 'laion400m_e31'),
 ('ViT-B-32', 'laion400m_e32'),
 ('ViT-B-32', 'laion2b_e16'),
 ('ViT-B-32', 'laion2b_s34b_b79k'),
 ('ViT-B-32', 'datacomp_xl_s13b_b90k'),
 ('ViT-B-32', 'datacomp_m_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_clip_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_laion_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_image_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_text_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_basic_s128m_b4k'),
 ('ViT-B-32', 'commonpool_m_s128m_b4k'),
 ('ViT-B-32', 'datacomp_s_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_clip_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_laion_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_image_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_text_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_basic_s13m_b4k'),
 ('ViT-B-32', 'commonpool_s_s13m_b4k'),
 ('ViT-

In [3]:
# load PDF
PDF_PATH = "../data/crag_paper.pdf"

loader = UnstructuredLoader(
    PDF_PATH,
    mode="elements",
    strategy="hi_res",
    extract_images_in_pdf=True,
)
elements = loader.load()

print(f"Loaded {len(elements)} elements")
for cat in sorted(set(el.metadata.get("category", "unknown") for el in elements)):
    count = sum(1 for el in elements if el.metadata.get("category") == cat)
    print(f"  {cat}: {count}")

INFO: pikepdf C++ to Python logger bridge initialized
INFO: HTTP Request: HEAD https://huggingface.co/unstructuredio/yolo_x_layout/resolve/main/yolox_l0.05.onnx "HTTP/1.1 302 Found"
INFO: Reading PDF for file: ../data/crag_paper.pdf ...


Loaded 255 elements
  FigureCaption: 6
  Footer: 1
  Formula: 1
  Header: 1
  Image: 6
  ListItem: 41
  NarrativeText: 107
  Table: 7
  Title: 32
  UncategorizedText: 53


In [4]:
# split text
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

text_elements = [el for el in elements if el.metadata.get("category") != "Image"]
text_docs = splitter.split_documents(text_elements)
print(f"Text chunks: {len(text_docs)}")

Text chunks: 258


In [5]:
# CLIP embeddings
clip_embeddings = OpenCLIPEmbeddings(model_name="ViT-B-32", checkpoint="laion2b_s34b_b79k")

INFO: Parsing model identifier. Schema: None, Identifier: ViT-B-32
INFO: Loaded built-in ViT-B-32 model config.
INFO: HTTP Request: HEAD https://huggingface.co/laion/CLIP-ViT-B-32-laion2B-s34B-b79K/resolve/main/open_clip_model.safetensors "HTTP/1.1 302 Found"
INFO: HTTP Request: GET https://huggingface.co/api/models/laion/CLIP-ViT-B-32-laion2B-s34B-b79K/xet-read-token/1a25a446712ba5ee05982a381eed697ef9b435cf "HTTP/1.1 200 OK"
INFO: Instantiating model architecture: CLIP
INFO: Loading full pretrained weights from: /home/rikesh/.cache/huggingface/hub/models--laion--CLIP-ViT-B-32-laion2B-s34B-b79K/snapshots/1a25a446712ba5ee05982a381eed697ef9b435cf/open_clip_model.safetensors
INFO: Final image preprocessing configuration set: {'size': (224, 224), 'mode': 'RGB', 'mean': (0.48145466, 0.4578275, 0.40821073), 'std': (0.26862954, 0.26130258, 0.27577711), 'interpolation': 'bicubic', 'resize_mode': 'shortest', 'fill_color': 0}
INFO: Model ViT-B-32 creation process complete.
INFO: Parsing tokenize

In [6]:
# vector store
store = Chroma(embedding_function=clip_embeddings, collection_name="multimodal")
text_ids = store.add_documents(filter_complex_metadata(text_docs))
print(f"Added {len(text_ids)} text chunks")

Added 258 text chunks


In [7]:
# add images
image_elements = [
    el for el in elements
    if el.metadata.get("category") == "Image" and el.metadata.get("image_path")
]

image_uris = [el.metadata["image_path"] for el in image_elements]
image_metadatas = [doc.metadata for doc in filter_complex_metadata(image_elements)]

image_ids = store.add_images(uris=image_uris, metadatas=image_metadatas)
print(f"Added {len(image_ids)} images")

Added 6 images


In [8]:
image_ids

['038fb9f4-842a-4d83-bcbb-1351a7a7672e',
 '3e2a0abd-ffbc-4e5b-9d44-93d529ccea72',
 '791f5977-6894-4490-ae3a-a1c340cb0d76',
 'f84dac10-e626-4536-b401-2a08417ad8e8',
 'b9a473ff-1c6f-4ac4-b282-525ab7cb84f1',
 'f7758cfa-93de-4a35-8b19-ea8b0cad65e3']

In [9]:
# retriever
retriever = store.as_retriever(search_kwargs={"k": 6})

In [10]:
# RAG chain
def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def build_messages(inputs):
    docs = inputs["docs"]
    question = inputs["question"]

    text_chunks = [d for d in docs if d.metadata.get("category") != "Image"]
    image_chunks = [d for d in docs if d.metadata.get("category") == "Image"]

    text_context = "\n\n".join(d.page_content for d in text_chunks)

    content = [
        {"type": "text", "text": f"Answer the question based only on the following context:\n\n{text_context}\n\nQuestion: {question}"},
    ]

    seen = set()
    for doc in image_chunks:
        path = doc.page_content  # add_images stores URI as page_content
        if path and path not in seen:
            seen.add(path)
            content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{encode_image(path)}"}})

    return [HumanMessage(content=content)]

llm = ChatOpenAI(model="gpt-5-mini")

chain = (
    {"docs": retriever, "question": RunnablePassthrough()}
    | RunnableLambda(build_messages)
    | llm
    | StrOutputParser()
)

In [11]:
# test
question = "How does generation accuracy change as retrieval accuracy drops for Self-CRAG and Self-RAG. explain in a detailed way?"
answer = chain.invoke(question)
print(answer)

INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


In the experiment described, the authors simulated a weaker retriever by randomly removing some of the correct retrieved passages and measured how generation accuracy changed for Self-RAG (the baseline) and Self-CRAG (their corrective plug-in) on PopQA.

What happened
- For both systems, generation accuracy declined as retrieval accuracy was reduced. This shows the generator depends heavily on having correct retrieved evidence — when useful passages are missing, the model has less factual context and makes more errors.
- The decline was steeper for Self-RAG than for Self-CRAG. In other words, Self-CRAG’s generation accuracy dropped more gently as retrieval quality worsened.

Why this occurs (detailed explanation)
- Missing or incorrect retrievals remove or corrupt the evidence the generator uses to answer questions, so generation performance falls in both systems.
- Self-CRAG mitigates that sensitivity because its corrective mechanism (the CRAG component) reduces the generator’s relian

In [12]:
question = "Computational requirements of CRAG vs Self-RAG and which was has faster execution time and can you give me the actual TFLOPS values?"
answer = chain.invoke(question)
print(answer)

INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


From the table in the context:

- CRAG: 27.2 TFLOPs per token; execution time 0.512 s per token.  
- Self-RAG: 26.5–132.4 TFLOPs per token (a wide range); execution time 0.741 s per token.

Thus CRAG has the faster execution time (0.512 s vs 0.741 s) and a fixed TFLOPs requirement of 27.2, while Self-RAG shows a variable TFLOPs cost between 26.5 and 132.4.
